# PDF Summarizer Agent — on-demand inference server

Docker/Gradio HF Spaces on the free CPU tier now require an HF PRO
subscription (confirmed 2026-08-18 — Static Spaces are the only free SDK).
So the shipped agent's inference runs the same way training does: a
manually-opened Kaggle/Colab session, this time exposed over a public
tunnel so the local `pdfsum` CLI can reach it.

**This is meant to be opened on demand, not left running.** Run it when you
want to summarize a PDF, copy the tunnel URL into the local CLI, use it,
then stop this notebook's session (Kaggle: stop the session; Colab:
Runtime > Disconnect) — an idle GPU session still burns the free weekly
GPU-hour quota, and Kaggle in particular has no API to force-stop a
forgotten one, capping the account at 2 concurrent GPU sessions.

**MANUAL_ACTION_REQUIRED before running:**
1. Open on Kaggle or Colab. GPU is optional but much faster — attach one
   if you have quota to spare for this session.
2. No secrets needed for the placeholder model below. Once the real
   fine-tuned model is ready (private HF repo), add an `HF_TOKEN` secret
   the same way as in the training notebook.
3. Run cells top to bottom, then copy the printed tunnel URL into:
   `pdfsum summarize <pdf> --endpoint <tunnel-url>`

In [ ]:
!pip install -q -U llama-cpp-python fastapi uvicorn huggingface_hub nest_asyncio
!wget -q -O cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared
# cloudflared quick tunnels need no account/token — the most $0-friendly
# tunnel option, unlike ngrok which requires a free-tier signup + authtoken.

In [ ]:
import os

# Placeholder model — swap once training (training/qlora_train.ipynb) has
# produced a real adapter and it's been merged + quantized to GGUF and
# pushed to a model repo. That merge/quantize step doesn't exist yet.
MODEL_REPO = os.environ.get("MODEL_REPO", "Qwen/Qwen2.5-0.5B-Instruct-GGUF")
MODEL_FILE = os.environ.get("MODEL_FILE", "qwen2.5-0.5b-instruct-q4_k_m.gguf")

hf_token = os.environ.get("HF_TOKEN")
if not hf_token:
    try:
        from kaggle_secrets import UserSecretsClient
        hf_token = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        try:
            from google.colab import userdata
            hf_token = userdata.get("HF_TOKEN")
        except Exception:
            pass  # fine for the public placeholder model above

In [ ]:
from huggingface_hub import hf_hub_download
from llama_cpp import Llama

model_path = hf_hub_download(repo_id=MODEL_REPO, filename=MODEL_FILE, token=hf_token)

# n_gpu_layers=-1 offloads everything to GPU if one is attached and the
# CUDA-enabled wheel is present; falls back to CPU otherwise. Don't trust a
# health check alone as proof it's on GPU — check this load log for
# "device CUDA0" vs "device CPU" (lesson from the sibling project).
llm = Llama(model_path=model_path, n_ctx=4096, n_gpu_layers=-1, verbose=True)

In [ ]:
import threading

import nest_asyncio
import uvicorn
from fastapi import FastAPI
from pydantic import BaseModel

nest_asyncio.apply()
app = FastAPI()


class SummarizeRequest(BaseModel):
    text: str
    max_tokens: int = 512


@app.get("/health")
def health():
    return {"status": "ok", "model_repo": MODEL_REPO, "model_file": MODEL_FILE}


@app.post("/summarize")
def summarize(req: SummarizeRequest):
    prompt = (
        "Summarize this document. Respond with ONLY a JSON object, no prose, "
        f"no markdown fences.\n\nDOCUMENT:\n{req.text}"
    )
    result = llm.create_chat_completion(
        messages=[{"role": "user", "content": prompt}], max_tokens=req.max_tokens,
    )
    return {"output": result["choices"][0]["message"]["content"]}


def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="warning")


threading.Thread(target=run_server, daemon=True).start()
print("local server started on :8000")

In [ ]:
import re
import subprocess
import time

# Fail-fast polling instead of a blind sleep, so a broken tunnel surfaces in
# ~30s rather than hanging silently (lesson from the sibling project).
tunnel = subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", "http://localhost:8000"],
    stderr=subprocess.PIPE, stdout=subprocess.PIPE, text=True,
)

public_url = None
deadline = time.time() + 30
while time.time() < deadline:
    line = tunnel.stderr.readline()
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if match:
        public_url = match.group(0)
        break

assert public_url, "tunnel did not report a URL within 30s — check cloudflared's output above"
print(f"PUBLIC ENDPOINT: {public_url}")
print(f"use it locally as: pdfsum summarize <pdf> --endpoint {public_url}")